# DIMER Workshop: Comparing Time-Series Foundation Models

**Profile:** `TASK-INFERENCE`  
**Mode:** `WORKSHOP`  
**Notebook specification:** `2.1`  
**Status:** Candidate workshop carrier  
**Default execution tier:** `STANDARD`  
**Recommended runtime:** NVIDIA Tesla T4 or equivalent

This standalone workshop compares multiple live DIMER time-series foundation models under **one common forecasting protocol**:

- **TiRex-2** — `NX-AI/TiRex-2`
- **Chronos-2** — `amazon/chronos-2`
- **Toto 2.0 2.5B** — `Datadog/Toto-2.0-2.5B` (`FULL` tier)

The notebook asks:

> Given exactly the same historical series, forecast horizon, baselines, quantiles, and evaluation periods, how do different pretrained forecasting models behave?

It does **not** declare a universally best model. The built-in sample is deterministic synthetic teaching data; all measured results are **tutorial/sanity evidence**, not benchmark evidence.

### Standalone contract

The canonical path:

- does not clone a Git repository;
- does not download or import DIMER repository source;
- does not call a DIMER worker, API, gateway, or control plane;
- does not require a token, login, or secret;
- automatically generates and digest-verifies the workshop sample;
- validates chronology before model acquisition;
- runs model inference locally through pinned upstream Python packages;
- freezes the experiment before independent test evaluation;
- exports machine-readable forecasts, metrics, and provenance; and
- provides optional BYOD without blocking **Run all**.

### Execution tiers

`STANDARD` — TiRex-2 + Chronos-2. This is the default release path.

`FULL` — TiRex-2 + Chronos-2 + Toto 2.0 2.5B. Toto adds a 9.8 GB checkpoint and requires a CUDA GPU.

### Learning goals

By the end of the workshop you should be able to:

1. create a leakage-safe chronological forecasting experiment;
2. distinguish context from forecast horizon;
3. compare foundation-model forecasts with last-value and seasonal-naive baselines;
4. interpret MAE, RMSE, baseline-relative skill, pinball loss, interval coverage, and interval width;
5. distinguish model quantiles from calibrated confidence intervals;
6. freeze model/evaluation choices before viewing an independent test period;
7. compare predictive quality with model size and runtime cost; and
8. repeat the workflow with your own regularly sampled numerical time series.

## 0. Prerequisites

The built-in fixture is the existing synthetic multi-series sample from the DIMER Chronos-2 pipeline repository's `examples/sample-data/generate_samples.py`, reproduced here as code so the notebook remains standalone.

Canonical identity:

- artifact: `chronos_multi_series.csv`
- 2 series (`A`, `B`)
- 96 hourly observations per series
- 192 long-format rows
- Apache-2.0 synthetic teaching data
- canonical SHA-256:  
  `6f12f1475411aaab13a2de860ba50f008bd7185a9266f99147c244db4a5de8b3`

The sample contains deterministic trend and 24-hour seasonality. It is intentionally simple enough that a **24-hour seasonal-naive baseline can be strong**. That is a feature of the workshop: sophisticated models must earn their complexity.

`STANDARD` downloads roughly 381 MB of TiRex-2 weights and 478 MB of Chronos-2 weights, in addition to package dependencies.

`FULL` additionally downloads Toto 2.0's 9,817,176,960-byte SafeTensors checkpoint.

## 1. Runtime and workshop controls

In [ ]:
# @title Workshop controls
WORKSHOP_TIER = "STANDARD"  # @param ["STANDARD", "FULL"]
USE_BYOD = False            # @param {type:"boolean"}
BYOD_CSV_PATH = ""          # @param {type:"string"}
OUTPUT_DIR = "outputs"      # @param {type:"string"}

VALIDATION_HORIZON = 24     # @param {type:"integer"}
TEST_HORIZON = 24           # @param {type:"integer"}
SEASON_LENGTH = 24          # @param {type:"integer"}

QUANTILES = (0.1, 0.5, 0.9)
COMMON_MAX_CONTEXT = 8192
COMMON_MAX_HORIZON = 1024
COMMON_MIN_CONTEXT = 32

if WORKSHOP_TIER not in {"STANDARD", "FULL"}:
    raise ValueError("WORKSHOP_TIER must be STANDARD or FULL")
if not 1 <= VALIDATION_HORIZON <= COMMON_MAX_HORIZON:
    raise ValueError("VALIDATION_HORIZON outside the common 1..1024 contract")
if not 1 <= TEST_HORIZON <= COMMON_MAX_HORIZON:
    raise ValueError("TEST_HORIZON outside the common 1..1024 contract")
if SEASON_LENGTH < 1:
    raise ValueError("SEASON_LENGTH must be positive")

SELECTED_MODELS = ["tirex", "chronos"] + (["toto"] if WORKSHOP_TIER == "FULL" else [])

from pathlib import Path
OUTPUT_ROOT = Path(OUTPUT_DIR)
WORK_ROOT = Path("work")
for path in [
    OUTPUT_ROOT / "data",
    OUTPUT_ROOT / "validation",
    OUTPUT_ROOT / "frozen",
    OUTPUT_ROOT / "test",
    OUTPUT_ROOT / "future",
    OUTPUT_ROOT / "figures",
    OUTPUT_ROOT / "provenance",
    WORK_ROOT / "envs",
    WORK_ROOT / "runners",
    WORK_ROOT / "model_cache",
    WORK_ROOT / "requests",
    WORK_ROOT / "run_metadata",
]:
    path.mkdir(parents=True, exist_ok=True)

print({
    "tier": WORKSHOP_TIER,
    "selected_models": SELECTED_MODELS,
    "validation_horizon": VALIDATION_HORIZON,
    "test_horizon": TEST_HORIZON,
    "season_length": SEASON_LENGTH,
    "output_root": str(OUTPUT_ROOT.resolve()),
})

In [ ]:
# @title Verify the parent notebook environment
import importlib.metadata
import platform
import sys

if sys.version_info[:2] < (3, 10):
    raise RuntimeError(
        f"The parent notebook runtime needs Python 3.10 or newer; "
        f"current runtime is {platform.python_version()}."
    )

# The heavy model stacks run in isolated virtual environments pinned to
# MODEL_ENV_PYTHON (built in Section "Build selected isolated environments"),
# so the parent kernel's own Python version does not constrain them. The parent
# runtime only owns orchestration, validation, tables and figures.
for package in ("numpy", "pandas", "matplotlib", "packaging"):
    try:
        print(package, importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError as exc:
        raise RuntimeError(f"Parent runtime is missing required package: {package}") from exc

print({
    "python": platform.python_version(),
    "platform": platform.platform(),
    "executable": sys.executable,
})

## 2. Model identities and common comparison boundary

The workshop uses the same immutable model identities and file digests recorded by the live DIMER pipeline profiles.

Only the **intersection** of all selected forecasting capabilities is used in the core comparison:

- finite numerical targets;
- regularly sampled histories;
- no exogenous covariates;
- context ≤ 8,192 observations;
- horizon ≤ 1,024 observations;
- q0.1 / q0.5 / q0.9;
- point forecast = q0.5 median.

TiRex and Chronos support additional covariate workflows, but giving them covariates while Toto receives none would make the comparison asymmetric. Covariates therefore belong only in optional model-specific experiments.

In [ ]:
MODEL_SPECS = {
    "tirex": {
        "display_name": "TiRex-2",
        "model_id": "NX-AI/TiRex-2",
        "revision": "05e5b26db52bfb256f1ae1bdf785589850482de3",
        "license": "Apache-2.0",
        "runtime_package": "tirex-2==0.2.1",
        "files": {
            "README.md": (5521, "ec10f277ff820fb8915b4ff3988916c25bc892167902aba84bb9bfe24d190b70"),
            "model-config.yaml": (1204, "cddbe6d0be4919cb7b0cad747fb19a295264e129d1c0790cf62133b4cb5da727"),
            "model.ckpt": (380613375, "184b160ffbe4c01a26beeba14015ff3507c7497e1f3577114187bbc1d19fcac1"),
        },
        "weight_bytes": 380613375,
        "device": "CPU",
    },
    "chronos": {
        "display_name": "Chronos-2",
        "model_id": "amazon/chronos-2",
        "revision": "95a9710e2596287d08352589f42634fa5abdf0a7",
        "license": "Apache-2.0",
        "runtime_package": "chronos-forecasting==2.3.1",
        "files": {
            "README.md": (31, "4bcf87ecfbbb8e07a01b21415a970c8b53a5283bf6872b657040d3f45c9241f7"),
            "config.json": (1067, "ef1143bfdc9c0376d9a056eefca46cb4b1ec3d0ffacd541ff56feb40fb708031"),
            "model.safetensors": (477930472, "ddcda3c7508bf2528087723e98a20707cc04b7f370ae275a9fd88078ddba4f42"),
        },
        "weight_bytes": 477930472,
        "device": "CPU/GPU",
    },
    "toto": {
        "display_name": "Toto 2.0 2.5B",
        "model_id": "Datadog/Toto-2.0-2.5B",
        "revision": "51a2812bbe449437c01b79c0e425ed578f335f5b",
        "license": "Apache-2.0",
        "runtime_package": "toto-2==2.0.0",
        "files": {
            "README.md": (7188, "5cd75f3d08d1574c085ee31a2886855d82247f76cb3bd0b8015750a7fc1d316d"),
            "config.json": (598, "172904c65bb5af77a95e1a81cdd9214f4c9d983651d6dceef1375d3d09f47a36"),
            "model.safetensors": (9817176960, "dc08942b20751ac906167194d4ca4aa06b4367e80aabe5f1d5153b30b874bdb9"),
        },
        "weight_bytes": 9817176960,
        "device": "CUDA GPU",
    },
}

import pandas as pd
display(pd.DataFrame([
    {
        "model": MODEL_SPECS[key]["display_name"],
        "model_id": MODEL_SPECS[key]["model_id"],
        "revision": MODEL_SPECS[key]["revision"][:12] + "…",
        "weights_MiB": round(MODEL_SPECS[key]["weight_bytes"] / 2**20, 1),
        "reference_device": MODEL_SPECS[key]["device"],
        "selected": key in SELECTED_MODELS,
    }
    for key in MODEL_SPECS
]))

## 3. Generate the canonical Chronos multi-series fixture or load BYOD

The built-in fixture is generated from fixed formulas—there is no random draw.

For `t = 0..95`:

**Series A**

`120 + 0.18*t + 6*sin(2*pi*t/24)`

**Series B**

`155 + 0.12*t + 9*sin(2*pi*(t+5)/24)`

The generated CSV bytes must reproduce the canonical repository digest before the workshop proceeds.

BYOD uses the same long schema:

```text
series_id,timestamp,target
A,2026-01-01T00:00:00,100.0
A,2026-01-01T01:00:00,101.0
...
B,2026-01-01T00:00:00,150.0
...
```

The default path never opens an upload dialog.

In [ ]:
# @title Generate or load the workshop panel
import csv
import hashlib
import io
import json
import os
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

CANONICAL_SAMPLE_SHA256 = "6f12f1475411aaab13a2de860ba50f008bd7185a9266f99147c244db4a5de8b3"

def canonical_sample_bytes():
    n = 96
    timestamps = pd.date_range("2026-01-01", periods=n, freq="h")
    step = np.arange(n, dtype=float)
    parts = []
    for series_id, offset, slope, amplitude, phase in (
        ("A", 0.0, 0.18, 6.0, 0.0),
        ("B", 35.0, 0.12, 9.0, 5.0),
    ):
        target = (
            120.0
            + offset
            + slope * step
            + amplitude * np.sin(2.0 * np.pi * (step + phase) / 24.0)
        )
        parts.append(pd.DataFrame({
            "series_id": series_id,
            "timestamp": timestamps,
            "target": target,
        }))
    frame = pd.concat(parts, ignore_index=True)
    payload = frame.to_csv(
        index=False,
        date_format="%Y-%m-%dT%H:%M:%S",
        float_format="%.4f",
        lineterminator="\n",
    ).encode("utf-8")
    return payload

def read_checked_csv(payload):
    rows = csv.reader(io.StringIO(payload.decode("utf-8-sig")))
    try:
        header = next(rows)
    except StopIteration as exc:
        raise ValueError("CSV is empty") from exc
    duplicates = sorted(name for name, count in Counter(header).items() if count > 1)
    if duplicates:
        raise ValueError(f"Duplicate CSV header(s) are ambiguous: {duplicates}")
    return pd.read_csv(io.BytesIO(payload))

if BYOD_CSV_PATH:
    payload = Path(BYOD_CSV_PATH).read_bytes()
    sample_kind = "BYOD"
    input_source = f"BYOD path: {BYOD_CSV_PATH}"
elif USE_BYOD:
    try:
        from google.colab import files
    except ImportError as exc:
        raise ValueError("Set BYOD_CSV_PATH when google.colab upload is unavailable") from exc
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV")
    uploaded_name, payload = next(iter(uploaded.items()))
    sample_kind = "BYOD"
    input_source = f"BYOD upload: {uploaded_name}"
else:
    payload = canonical_sample_bytes()
    observed = hashlib.sha256(payload).hexdigest()
    if observed != CANONICAL_SAMPLE_SHA256:
        raise RuntimeError(
            f"Canonical sample digest mismatch: {observed} != {CANONICAL_SAMPLE_SHA256}"
        )
    sample_kind = "synthetic"
    input_source = "inline generator equivalent to chronos_multi_series.csv"

frame = read_checked_csv(payload)
input_sha256 = hashlib.sha256(payload).hexdigest()
frame.to_csv(OUTPUT_ROOT / "data" / "validated_panel_source.csv", index=False)

print({
    "sample_kind": sample_kind,
    "source": input_source,
    "sha256": input_sha256,
    "rows": len(frame),
    "columns": list(frame.columns),
})
display(frame.head())

## 4. Validate chronology before any model download

In [ ]:
# @title Validate the common forecasting input contract
REQUIRED_COLUMNS = ["series_id", "timestamp", "target"]

def validate_panel(frame):
    missing = [c for c in REQUIRED_COLUMNS if c not in frame.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    work = frame[REQUIRED_COLUMNS].copy()

    if work["series_id"].isna().any():
        raise ValueError("series_id contains null values")
    work["series_id"] = work["series_id"].astype(str)
    if (work["series_id"].str.len() == 0).any():
        raise ValueError("series_id contains empty values")

    work["timestamp"] = pd.to_datetime(work["timestamp"], errors="raise")
    work["target"] = pd.to_numeric(work["target"], errors="raise")
    if not np.isfinite(work["target"].to_numpy(dtype=float)).all():
        raise ValueError("target contains missing or non-finite values")

    if work.duplicated(["series_id", "timestamp"]).any():
        raise ValueError("Duplicate (series_id, timestamp) observations are not allowed")

    work = work.sort_values(["series_id", "timestamp"], kind="mergesort").reset_index(drop=True)

    ids = list(work["series_id"].drop_duplicates())
    if not ids:
        raise ValueError("No target series found")
    if len(ids) > 16:
        raise ValueError("Workshop BYOD is limited to 16 series")

    time_grids = []
    for sid in ids:
        block = work[work["series_id"] == sid]
        ts = pd.DatetimeIndex(block["timestamp"])
        if not ts.is_monotonic_increasing or ts.has_duplicates:
            raise ValueError(f"{sid}: timestamps must be strictly increasing and unique")
        if len(ts) < 2:
            raise ValueError(f"{sid}: at least two timestamps are required")
        deltas = pd.Series(ts).diff().dropna()
        if deltas.nunique() != 1:
            raise ValueError(f"{sid}: irregular or gappy frequency is not supported")
        time_grids.append(ts)

    reference = time_grids[0]
    for sid, grid in zip(ids[1:], time_grids[1:]):
        if not grid.equals(reference):
            raise ValueError(
                f"{sid}: all series must use exactly the same timestamp grid in this comparative workshop"
            )

    delta = reference[1] - reference[0]
    try:
        freq_alias = pd.tseries.frequencies.to_offset(delta).freqstr
    except Exception as exc:
        raise ValueError(f"Could not represent fixed-width frequency {delta}") from exc

    n_steps = len(reference)
    required_tail = VALIDATION_HORIZON + TEST_HORIZON
    if n_steps - required_tail < COMMON_MIN_CONTEXT:
        raise ValueError(
            f"Need at least {COMMON_MIN_CONTEXT + required_tail} observations per series "
            f"({COMMON_MIN_CONTEXT} context + {required_tail} holdout); got {n_steps}"
        )
    if n_steps - TEST_HORIZON > COMMON_MAX_CONTEXT:
        print(
            f"NOTE: test history has {n_steps - TEST_HORIZON} observations; "
            f"the common comparison will keep only the most recent {COMMON_MAX_CONTEXT}."
        )

    return {
        "frame": work,
        "series_ids": ids,
        "timestamps": reference,
        "frequency": delta,
        "frequency_alias": freq_alias,
        "n_steps": n_steps,
        "n_series": len(ids),
        "target_min": float(work["target"].min()),
        "target_max": float(work["target"].max()),
    }

panel = validate_panel(frame)
frame = panel["frame"]

dataset_manifest = {
    "sample_kind": sample_kind,
    "source": input_source,
    "sha256": input_sha256,
    "license": "Apache-2.0" if sample_kind == "synthetic" else "user-supplied; caller-owned",
    "n_series": panel["n_series"],
    "series_ids": panel["series_ids"],
    "n_steps_per_series": panel["n_steps"],
    "frequency": str(panel["frequency"]),
    "frequency_alias": panel["frequency_alias"],
    "validation_horizon": VALIDATION_HORIZON,
    "test_horizon": TEST_HORIZON,
    "season_length": SEASON_LENGTH,
}
(OUTPUT_ROOT / "data" / "dataset_manifest.json").write_text(
    json.dumps(dataset_manifest, indent=2), encoding="utf-8"
)

print(json.dumps(dataset_manifest, indent=2))

## 5. Explore the panel and locate the forecasting boundaries

For the built-in sample:

- hours 0–47 are the initial historical context;
- hours 48–71 are the validation future;
- hours 72–95 are the independent test future.

For BYOD, the same rule is applied from the tail: reserve the final `TEST_HORIZON`, reserve the preceding `VALIDATION_HORIZON`, and use the earlier observations as validation context.

The final test forecast is allowed to use the former validation observations because, at the later forecast origin, they are now legitimately in the past.

In [ ]:
# @title Visualize the series, seasonality, and split boundaries
import matplotlib.pyplot as plt

val_start_index = panel["n_steps"] - VALIDATION_HORIZON - TEST_HORIZON
test_start_index = panel["n_steps"] - TEST_HORIZON

fig, ax = plt.subplots(figsize=(12, 5))
for sid in panel["series_ids"]:
    block = frame[frame["series_id"] == sid]
    ax.plot(block["timestamp"], block["target"], label=sid)
ax.axvline(panel["timestamps"][val_start_index], linestyle="--", label="validation starts")
ax.axvline(panel["timestamps"][test_start_index], linestyle="--", label="test starts")
ax.set_title("Workshop time-series panel")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Target")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "figures" / "panel_overview.png", dpi=150, bbox_inches="tight")
plt.show()

# Normalize only for visualization; models receive raw target values.
fig, ax = plt.subplots(figsize=(12, 4))
for sid in panel["series_ids"]:
    block = frame[frame["series_id"] == sid].copy()
    values = block["target"].to_numpy(dtype=float)
    z = (values - values.mean()) / (values.std() or 1.0)
    ax.plot(block["timestamp"], z, label=sid)
ax.set_title("Normalized overlay for visual comparison only")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Standardized target")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "figures" / "normalized_overlay.png", dpi=150, bbox_inches="tight")
plt.show()

display(frame.groupby("series_id")["target"].agg(["count", "mean", "std", "min", "max"]))

### Workshop prediction

Before running the foundation models:

> Which baseline should perform better on this particular synthetic fixture: last value or the 24-hour seasonal naive?

The answer should follow from the visible 24-hour periodic structure. A large pretrained model is not automatically useful simply because it is more complex.

## 6. Common split, baselines, and model-independent evaluator

In [ ]:
# @title Create validation/test requests and common metric functions
def tail_context(block, end_index, max_context=COMMON_MAX_CONTEXT):
    start = max(0, end_index - max_context)
    return block.iloc[start:end_index].copy()

def split_stage(frame, stage):
    ids = panel["series_ids"]
    pieces_history, pieces_truth = [], []
    n = panel["n_steps"]
    if stage == "validation":
        truth_start = n - TEST_HORIZON - VALIDATION_HORIZON
        truth_horizon = VALIDATION_HORIZON
    elif stage == "test":
        truth_start = n - TEST_HORIZON
        truth_horizon = TEST_HORIZON
    elif stage == "future":
        truth_start = n
        truth_horizon = TEST_HORIZON
    else:
        raise ValueError(stage)

    for sid in ids:
        block = frame[frame["series_id"] == sid].sort_values("timestamp").reset_index(drop=True)
        hist = tail_context(block, truth_start)
        if len(hist) < COMMON_MIN_CONTEXT:
            raise ValueError(f"{stage}/{sid}: context shorter than {COMMON_MIN_CONTEXT}")
        pieces_history.append(hist)
        if stage != "future":
            pieces_truth.append(block.iloc[truth_start:truth_start + truth_horizon].copy())

    history = pd.concat(pieces_history, ignore_index=True)
    truth = pd.concat(pieces_truth, ignore_index=True) if pieces_truth else None
    return history, truth, truth_horizon

validation_history, validation_truth, _ = split_stage(frame, "validation")
test_history, test_truth, _ = split_stage(frame, "test")
future_history, _, future_horizon = split_stage(frame, "future")

for name, hist, truth in (
    ("validation", validation_history, validation_truth),
    ("test", test_history, test_truth),
):
    for sid in panel["series_ids"]:
        h = hist[hist["series_id"] == sid]
        t = truth[truth["series_id"] == sid]
        assert h["timestamp"].max() < t["timestamp"].min(), f"{name}/{sid}: leakage"

validation_history.to_csv(WORK_ROOT / "requests" / "validation_history.csv", index=False)
test_history.to_csv(WORK_ROOT / "requests" / "test_history.csv", index=False)
future_history.to_csv(WORK_ROOT / "requests" / "future_history.csv", index=False)

def mae(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return float(np.mean(np.abs(y - p)))

def rmse(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return float(np.sqrt(np.mean((y - p) ** 2)))

def pinball_loss(y, qhat, q):
    y, qhat = np.asarray(y, float), np.asarray(qhat, float)
    error = y - qhat
    return float(np.mean(np.maximum(q * error, (q - 1.0) * error)))

def baseline_forecast(history, truth, kind, season_length=SEASON_LENGTH):
    rows = []
    for sid in panel["series_ids"]:
        h = history[history["series_id"] == sid].sort_values("timestamp")
        t = truth[truth["series_id"] == sid].sort_values("timestamp")
        values = h["target"].to_numpy(dtype=float)
        if kind == "last_value":
            pred = np.repeat(values[-1], len(t))
        elif kind == "seasonal_naive":
            if len(values) < season_length:
                raise ValueError(
                    f"{sid}: seasonal baseline needs at least {season_length} historical observations"
                )
            repeats = []
            for step in range(len(t)):
                repeats.append(values[-season_length + (step % season_length)])
            pred = np.asarray(repeats, dtype=float)
        else:
            raise ValueError(kind)
        for step, ((_, truth_row), forecast) in enumerate(zip(t.iterrows(), pred), start=1):
            rows.append({
                "model": kind,
                "series_id": sid,
                "timestamp": truth_row["timestamp"],
                "step": step,
                "prediction": float(forecast),
                "q0.1": float(forecast),
                "q0.5": float(forecast),
                "q0.9": float(forecast),
            })
    return pd.DataFrame(rows)

def evaluate_forecasts(forecasts, truth, seasonal_baseline):
    truth_key = truth[["series_id", "timestamp", "target"]].rename(columns={"target": "truth"})
    merged = forecasts.merge(truth_key, on=["series_id", "timestamp"], how="left", validate="one_to_one")
    if merged["truth"].isna().any():
        raise ValueError("Forecast rows do not align exactly to held-out truth")

    seasonal = seasonal_baseline.merge(
        truth_key, on=["series_id", "timestamp"], how="left", validate="one_to_one"
    )
    seasonal_mae_by_series = {
        sid: mae(block["truth"], block["prediction"])
        for sid, block in seasonal.groupby("series_id")
    }

    per_series = []
    for (model, sid), block in merged.groupby(["model", "series_id"], sort=False):
        model_mae = mae(block["truth"], block["prediction"])
        seasonal_mae = seasonal_mae_by_series[sid]
        per_series.append({
            "model": model,
            "series_id": sid,
            "mae": model_mae,
            "rmse": rmse(block["truth"], block["prediction"]),
            "seasonal_naive_mae": seasonal_mae,
            "mae_skill_vs_seasonal": (
                1.0 - model_mae / seasonal_mae if seasonal_mae > 0 else None
            ),
            "pinball_q0.1": pinball_loss(block["truth"], block["q0.1"], 0.1),
            "pinball_q0.5": pinball_loss(block["truth"], block["q0.5"], 0.5),
            "pinball_q0.9": pinball_loss(block["truth"], block["q0.9"], 0.9),
            "q10_q90_coverage": float(
                np.mean(
                    (block["truth"].to_numpy() >= block["q0.1"].to_numpy())
                    & (block["truth"].to_numpy() <= block["q0.9"].to_numpy())
                )
            ),
            "mean_interval_width": float(
                np.mean(block["q0.9"].to_numpy() - block["q0.1"].to_numpy())
            ),
        })
    per_series = pd.DataFrame(per_series)

    aggregate_rows = []
    for model, block in merged.groupby("model", sort=False):
        ps = per_series[per_series["model"] == model]
        aggregate_rows.append({
            "model": model,
            "pooled_mae": mae(block["truth"], block["prediction"]),
            "pooled_rmse": rmse(block["truth"], block["prediction"]),
            "macro_mae": float(ps["mae"].mean()),
            "macro_rmse": float(ps["rmse"].mean()),
            "macro_mae_skill_vs_seasonal": float(ps["mae_skill_vs_seasonal"].mean()),
            "q10_q90_coverage": float(
                np.mean(
                    (block["truth"].to_numpy() >= block["q0.1"].to_numpy())
                    & (block["truth"].to_numpy() <= block["q0.9"].to_numpy())
                )
            ),
            "mean_interval_width": float(
                np.mean(block["q0.9"].to_numpy() - block["q0.1"].to_numpy())
            ),
            "pinball_q0.1": pinball_loss(block["truth"], block["q0.1"], 0.1),
            "pinball_q0.5": pinball_loss(block["truth"], block["q0.5"], 0.5),
            "pinball_q0.9": pinball_loss(block["truth"], block["q0.9"], 0.9),
        })
    return merged, per_series, pd.DataFrame(aggregate_rows)

validation_last = baseline_forecast(validation_history, validation_truth, "last_value")
validation_seasonal = baseline_forecast(validation_history, validation_truth, "seasonal_naive")
test_last = baseline_forecast(test_history, test_truth, "last_value")
test_seasonal = baseline_forecast(test_history, test_truth, "seasonal_naive")

validation_last.to_csv(OUTPUT_ROOT / "validation" / "last_value.csv", index=False)
validation_seasonal.to_csv(OUTPUT_ROOT / "validation" / "seasonal_naive.csv", index=False)
test_last.to_csv(OUTPUT_ROOT / "test" / "last_value.csv", index=False)
test_seasonal.to_csv(OUTPUT_ROOT / "test" / "seasonal_naive.csv", index=False)

print({
    "validation_context_steps": len(validation_history) // panel["n_series"],
    "validation_horizon": VALIDATION_HORIZON,
    "test_context_steps": len(test_history) // panel["n_series"],
    "test_horizon": TEST_HORIZON,
})

## 7. Isolated model environments

The three model packages use materially different pinned dependency stacks. The workshop therefore creates one virtual environment per model instead of mutating the notebook kernel.

This avoids manual runtime restarts and lets each adapter reproduce the package versions already qualified by its DIMER pipeline repository.

The environments are ordinary local Python virtual environments. No DIMER package is installed.

In [13]:
# @title Build selected isolated environments
import os
import subprocess
import sys
from pathlib import Path

# Model environments are pinned to Python 3.12 regardless of the notebook kernel
# (Colab moved to Python 3.13). uv provisions a standalone CPython 3.12 when the
# runtime does not have one.
MODEL_ENV_PYTHON = "3.12"
UV_PIN = "uv==0.8.17"

def ensure_uv():
    try:
        subprocess.run([sys.executable, "-m", "uv", "--version"], check=True, capture_output=True)
    except (subprocess.CalledProcessError, FileNotFoundError):
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", UV_PIN], check=True)

ENV_PINS = {
    "tirex": [
        "tirex-2==0.2.1",
        "torch==2.8.0",
        "torchvision==0.23.0",
        "torchaudio==2.8.0",
        "huggingface-hub==0.36.2",
        "numpy==2.3.3",
        "pandas==2.3.3",
    ],
    "chronos": [
        "chronos-forecasting==2.3.1",
        "transformers==4.57.6",
        "huggingface-hub==0.36.2",
        "numpy==2.5.3",
        "pandas==3.0.5",
        "torch==2.14.0",
        "torchvision==0.29.0",
        "torchaudio==2.11.0",
    ],
    "toto": [
        "toto-2==2.0.0",
        "torch==2.7.0",
        "torchaudio==2.7.0",
        "torchvision==0.22.0",
        "huggingface-hub==0.36.2",
        "transformers==4.57.6",
        "numpy==1.26.4",
        "pandas==2.2.3",
    ],
}

def env_python(name):
    root = WORK_ROOT / "envs" / name
    if os.name == "nt":
        return root / "Scripts" / "python.exe"
    return root / "bin" / "python"

def ensure_environment(name):
    root = WORK_ROOT / "envs" / name
    python = env_python(name)
    marker = root / ".dimer_workshop_ready"
    signature = "\n".join([f"python=={MODEL_ENV_PYTHON}", *ENV_PINS[name]])
    sig = __import__("hashlib").sha256(signature.encode()).hexdigest()
    if marker.is_file() and marker.read_text().strip() == sig and python.is_file():
        print(f"{name}: existing pinned environment reused")
        return python

    if root.exists():
        import shutil
        shutil.rmtree(root)
    subprocess.run(
        [sys.executable, "-m", "uv", "venv", "--seed", "--python", MODEL_ENV_PYTHON, str(root)],
        check=True,
    )
    subprocess.run(
        [str(python), "-m", "pip", "install", "--quiet", "--upgrade", "pip"],
        check=True,
    )
    subprocess.run(
        [str(python), "-m", "pip", "install", "--quiet", *ENV_PINS[name]],
        check=True,
    )
    marker.write_text(sig, encoding="utf-8")
    print(f"{name}: environment ready")
    return python

ensure_uv()
MODEL_PYTHONS = {name: ensure_environment(name) for name in SELECTED_MODELS}
print({k: str(v) for k, v in MODEL_PYTHONS.items()})

tirex: environment ready
chronos: environment ready
{'tirex': 'work/envs/tirex/bin/python', 'chronos': 'work/envs/chronos/bin/python'}


## 8. Standalone upstream adapters

The following runner programs are carried directly inside this notebook.

They:

- download only the pinned upstream files;
- verify exact byte size and SHA-256;
- load through the upstream installed package;
- run zero-shot forecasting;
- normalize the result to the common q0.1/q0.5/q0.9 schema;
- record load/inference runtime; and
- write CSV + JSON results.

They do **not** download DIMER source or import a DIMER pipeline package.

In [14]:
# @title Write the three upstream adapter programs
import hashlib
from pathlib import Path

TIREX_RUNNER = r"""
import argparse, hashlib, json, platform, time
from pathlib import Path
import numpy as np
import pandas as pd

MODEL_ID="NX-AI/TiRex-2"
REVISION="05e5b26db52bfb256f1ae1bdf785589850482de3"
FILES={
 "README.md":(5521,"ec10f277ff820fb8915b4ff3988916c25bc892167902aba84bb9bfe24d190b70"),
 "model-config.yaml":(1204,"cddbe6d0be4919cb7b0cad747fb19a295264e129d1c0790cf62133b4cb5da727"),
 "model.ckpt":(380613375,"184b160ffbe4c01a26beeba14015ff3507c7497e1f3577114187bbc1d19fcac1"),
}
LEVELS=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]

def sha(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1<<20),b""): h.update(chunk)
    return h.hexdigest()

def stage(root):
    from huggingface_hub import hf_hub_download
    root.mkdir(parents=True,exist_ok=True)
    for name,(size,digest) in FILES.items():
        p=Path(hf_hub_download(MODEL_ID,name,revision=REVISION,local_dir=str(root)))
        if p.stat().st_size!=size or sha(p)!=digest:
            raise ValueError(f"{name}: integrity mismatch")
    return root

def main():
    ap=argparse.ArgumentParser()
    ap.add_argument("--input",required=True)
    ap.add_argument("--output",required=True)
    ap.add_argument("--metadata",required=True)
    ap.add_argument("--cache",required=True)
    ap.add_argument("--horizon",type=int,required=True)
    ap.add_argument("--partition",required=True)
    args=ap.parse_args()

    frame=pd.read_csv(args.input,parse_dates=["timestamp"])
    ids=list(frame["series_id"].astype(str).drop_duplicates())
    arrays=[]
    for sid in ids:
        block=frame[frame["series_id"].astype(str)==sid].sort_values("timestamp")
        arrays.append(block["target"].to_numpy(dtype=np.float32))
    lengths={len(x) for x in arrays}
    if len(lengths)!=1: raise ValueError("all series must have equal context length")
    values=np.stack(arrays)

    started=time.perf_counter()
    root=stage(Path(args.cache))
    from tirex2 import load_model, TimeseriesType
    import torch
    model=load_model(str(root),device="cpu")
    load_seconds=time.perf_counter()-started

    started=time.perf_counter()
    ts=TimeseriesType(target=torch.from_numpy(values))
    q=np.asarray(model.forecast([ts],prediction_length=args.horizon,output_type="numpy")[0],dtype=float)
    inference_seconds=time.perf_counter()-started
    expected=(len(ids),len(LEVELS),args.horizon)
    if q.shape!=expected: raise RuntimeError(f"unexpected shape {q.shape}, expected {expected}")

    delta=frame.sort_values("timestamp")["timestamp"].drop_duplicates().diff().dropna().iloc[0]
    origins={sid:frame[frame["series_id"].astype(str)==sid]["timestamp"].max() for sid in ids}
    rows=[]
    for i,sid in enumerate(ids):
        for step in range(args.horizon):
            rows.append({
                "model":"TiRex-2","partition":args.partition,
                "forecast_origin":origins[sid],
                "series_id":sid,"timestamp":origins[sid]+delta*(step+1),"step":step+1,
                "q0.1":q[i,0,step],"q0.5":q[i,4,step],"q0.9":q[i,8,step],
                "prediction":q[i,4,step],
            })
    out=pd.DataFrame(rows)
    if not np.allclose(out["prediction"],out["q0.5"],rtol=0,atol=0):
        raise RuntimeError("TiRex normalized prediction != q0.5")
    out.to_csv(args.output,index=False)
    Path(args.metadata).write_text(json.dumps({
        "model":"TiRex-2","model_id":MODEL_ID,"revision":REVISION,"device":"cpu",
        "context_length":int(values.shape[1]),"horizon":args.horizon,
        "load_seconds":load_seconds,"inference_seconds":inference_seconds,
        "weight_bytes":FILES["model.ckpt"][0],"python":platform.python_version()
    },indent=2),encoding="utf-8")

if __name__=="__main__": main()
"""

CHRONOS_RUNNER = r"""
import argparse, hashlib, json, platform, time
from pathlib import Path
import numpy as np
import pandas as pd

MODEL_ID="amazon/chronos-2"
REVISION="95a9710e2596287d08352589f42634fa5abdf0a7"
FILES={
 "README.md":(31,"4bcf87ecfbbb8e07a01b21415a970c8b53a5283bf6872b657040d3f45c9241f7"),
 "config.json":(1067,"ef1143bfdc9c0376d9a056eefca46cb4b1ec3d0ffacd541ff56feb40fb708031"),
 "model.safetensors":(477930472,"ddcda3c7508bf2528087723e98a20707cc04b7f370ae275a9fd88078ddba4f42"),
}

def sha(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1<<20),b""): h.update(chunk)
    return h.hexdigest()

def stage(root):
    from huggingface_hub import hf_hub_download
    root.mkdir(parents=True,exist_ok=True)
    for name,(size,digest) in FILES.items():
        p=Path(hf_hub_download(MODEL_ID,name,revision=REVISION,local_dir=str(root)))
        if p.stat().st_size!=size or sha(p)!=digest:
            raise ValueError(f"{name}: integrity mismatch")
    refused=list(root.glob("*.bin"))+list(root.glob("*.pt"))+list(root.glob("*.pth"))+list(root.glob("*.ckpt"))
    if refused: raise ValueError(f"refused pickle-style model files present: {refused}")
    return root

def main():
    ap=argparse.ArgumentParser()
    ap.add_argument("--input",required=True)
    ap.add_argument("--output",required=True)
    ap.add_argument("--metadata",required=True)
    ap.add_argument("--cache",required=True)
    ap.add_argument("--horizon",type=int,required=True)
    ap.add_argument("--partition",required=True)
    ap.add_argument("--freq",required=True)
    args=ap.parse_args()

    frame=pd.read_csv(args.input,parse_dates=["timestamp"])
    frame["series_id"]=frame["series_id"].astype(str)

    started=time.perf_counter()
    root=stage(Path(args.cache))
    import torch
    from chronos import BaseChronosPipeline
    device="cuda" if torch.cuda.is_available() else "cpu"
    pipe=BaseChronosPipeline.from_pretrained(str(root),device_map=device,dtype="auto")
    load_seconds=time.perf_counter()-started

    started=time.perf_counter()
    raw=pipe.predict_df(
        frame,
        future_df=None,
        id_column="series_id",
        timestamp_column="timestamp",
        target=["target"],
        prediction_length=args.horizon,
        quantile_levels=[0.1,0.5,0.9],
        batch_size=256,
        context_length=None,
        cross_learning=False,
        validate_inputs=True,
        freq=args.freq,
    )
    inference_seconds=time.perf_counter()-started

    required=["series_id","timestamp","target_name","predictions","0.1","0.5","0.9"]
    missing=[c for c in required if c not in raw.columns]
    if missing: raise RuntimeError(f"Chronos raw output missing {missing}; got {list(raw.columns)}")
    if set(raw["target_name"].astype(str))!={"target"}:
        raise RuntimeError("unexpected target_name values")
    out=raw.rename(columns={
        "predictions":"prediction","0.1":"q0.1","0.5":"q0.5","0.9":"q0.9"
    })[["series_id","timestamp","prediction","q0.1","q0.5","q0.9"]].copy()
    out["series_id"]=out["series_id"].astype(str)
    out["timestamp"]=pd.to_datetime(out["timestamp"])
    out["step"]=out.groupby("series_id",sort=False).cumcount()+1
    origins=frame.groupby("series_id")["timestamp"].max().to_dict()
    out["forecast_origin"]=out["series_id"].map(origins)
    out["model"]="Chronos-2"
    out["partition"]=args.partition
    out=out[["model","partition","forecast_origin","series_id","timestamp","step","q0.1","q0.5","q0.9","prediction"]]
    if not np.allclose(out["prediction"],out["q0.5"],rtol=0,atol=1e-6):
        raise RuntimeError("Chronos normalized prediction != q0.5")
    out.to_csv(args.output,index=False)
    Path(args.metadata).write_text(json.dumps({
        "model":"Chronos-2","model_id":MODEL_ID,"revision":REVISION,"device":device,
        "context_length":int(frame.groupby("series_id").size().max()),"horizon":args.horizon,
        "load_seconds":load_seconds,"inference_seconds":inference_seconds,
        "weight_bytes":FILES["model.safetensors"][0],"python":platform.python_version()
    },indent=2),encoding="utf-8")

if __name__=="__main__": main()
"""

TOTO_RUNNER = r"""
import argparse, hashlib, json, platform, time
from pathlib import Path
import numpy as np
import pandas as pd

MODEL_ID="Datadog/Toto-2.0-2.5B"
REVISION="51a2812bbe449437c01b79c0e425ed578f335f5b"
FILES={
 "README.md":(7188,"5cd75f3d08d1574c085ee31a2886855d82247f76cb3bd0b8015750a7fc1d316d"),
 "config.json":(598,"172904c65bb5af77a95e1a81cdd9214f4c9d983651d6dceef1375d3d09f47a36"),
 "model.safetensors":(9817176960,"dc08942b20751ac906167194d4ca4aa06b4367e80aabe5f1d5153b30b874bdb9"),
}
LEVELS=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]

def sha(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1<<20),b""): h.update(chunk)
    return h.hexdigest()

def stage(root):
    from huggingface_hub import hf_hub_download
    root.mkdir(parents=True,exist_ok=True)
    for name,(size,digest) in FILES.items():
        p=Path(hf_hub_download(MODEL_ID,name,revision=REVISION,local_dir=str(root)))
        if p.stat().st_size!=size or sha(p)!=digest:
            raise ValueError(f"{name}: integrity mismatch")
    return root

def main():
    ap=argparse.ArgumentParser()
    ap.add_argument("--input",required=True)
    ap.add_argument("--output",required=True)
    ap.add_argument("--metadata",required=True)
    ap.add_argument("--cache",required=True)
    ap.add_argument("--horizon",type=int,required=True)
    ap.add_argument("--partition",required=True)
    args=ap.parse_args()

    frame=pd.read_csv(args.input,parse_dates=["timestamp"])
    frame["series_id"]=frame["series_id"].astype(str)
    ids=list(frame["series_id"].drop_duplicates())
    arrays=[]
    for sid in ids:
        block=frame[frame["series_id"]==sid].sort_values("timestamp")
        arrays.append(block["target"].to_numpy(dtype=np.float32))
    if len({len(x) for x in arrays})!=1: raise ValueError("all series need equal context")
    values=np.stack(arrays)

    started=time.perf_counter()
    root=stage(Path(args.cache))
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Toto FULL tier requires a CUDA GPU")
    from toto2 import Toto2Model
    model=Toto2Model.from_pretrained(str(root),map_location="cpu").to("cuda").eval()
    load_seconds=time.perf_counter()-started

    patch_size=int(model.config.patch_size)
    if values.shape[1] < patch_size:
        raise ValueError(f"context shorter than patch size {patch_size}")
    padding=(patch_size-values.shape[1]%patch_size)%patch_size
    target=torch.as_tensor(values,dtype=torch.float32,device="cuda").unsqueeze(0)
    mask=torch.ones_like(target,dtype=torch.bool)
    if padding:
        target=torch.nn.functional.pad(target,(padding,0),value=0.0)
        mask=torch.nn.functional.pad(mask,(padding,0),value=False)
    series_ids=torch.arange(values.shape[0],device="cuda",dtype=torch.long).unsqueeze(0)

    started=time.perf_counter()
    with torch.inference_mode():
        forecast=model.forecast(
            {"target":target,"target_mask":mask,"series_ids":series_ids},
            horizon=args.horizon,
            decode_block_size=768,
            has_missing_values=padding>0,
        )
    inference_seconds=time.perf_counter()-started
    q=np.asarray(forecast.detach().cpu(),dtype=float)
    expected=(len(LEVELS),1,len(ids),args.horizon)
    if q.shape!=expected: raise RuntimeError(f"unexpected shape {q.shape}; expected {expected}")
    q=np.transpose(q[:,0,:,:],(1,0,2))

    delta=frame.sort_values("timestamp")["timestamp"].drop_duplicates().diff().dropna().iloc[0]
    origins={sid:frame[frame["series_id"]==sid]["timestamp"].max() for sid in ids}
    rows=[]
    for i,sid in enumerate(ids):
        for step in range(args.horizon):
            rows.append({
                "model":"Toto 2.0 2.5B","partition":args.partition,
                "forecast_origin":origins[sid],
                "series_id":sid,"timestamp":origins[sid]+delta*(step+1),"step":step+1,
                "q0.1":q[i,0,step],"q0.5":q[i,4,step],"q0.9":q[i,8,step],
                "prediction":q[i,4,step],
            })
    out=pd.DataFrame(rows)
    if not np.allclose(out["prediction"],out["q0.5"],rtol=0,atol=0):
        raise RuntimeError("Toto normalized prediction != q0.5")
    out.to_csv(args.output,index=False)
    Path(args.metadata).write_text(json.dumps({
        "model":"Toto 2.0 2.5B","model_id":MODEL_ID,"revision":REVISION,"device":"cuda",
        "context_length":int(values.shape[1]),"context_padding":int(padding),
        "patch_size":patch_size,"horizon":args.horizon,"decode_block_size":768,
        "load_seconds":load_seconds,"inference_seconds":inference_seconds,
        "weight_bytes":FILES["model.safetensors"][0],"python":platform.python_version()
    },indent=2),encoding="utf-8")

if __name__=="__main__": main()
"""

RUNNERS = {"tirex": TIREX_RUNNER, "chronos": CHRONOS_RUNNER, "toto": TOTO_RUNNER}

for name, source in RUNNERS.items():
    path = WORK_ROOT / "runners" / f"{name}_runner.py"
    path.write_text(source, encoding="utf-8")
    compile(source, str(path), "exec")
    print(name, path, hashlib.sha256(source.encode()).hexdigest()[:16] + "…")

tirex work/runners/tirex_runner.py 4256721e2bf1f2e9…
chronos work/runners/chronos_runner.py 7782bda512ae1f58…
toto work/runners/toto_runner.py 0f7bdc59d0ca5c56…


## 9. Run the common validation forecast

In [15]:
# @title Execute selected models on the validation future
import json
import os
import subprocess
import time

def run_model(name, stage, input_path, horizon):
    python = MODEL_PYTHONS[name]
    runner = WORK_ROOT / "runners" / f"{name}_runner.py"
    output = OUTPUT_ROOT / stage / f"{name}.csv"
    metadata = WORK_ROOT / "run_metadata" / f"{stage}_{name}.json"
    cache = WORK_ROOT / "model_cache" / name
    cmd = [
        str(python), str(runner),
        "--input", str(input_path),
        "--output", str(output),
        "--metadata", str(metadata),
        "--cache", str(cache),
        "--horizon", str(horizon),
        "--partition", stage,
    ]
    if name == "chronos":
        cmd += ["--freq", panel["frequency_alias"]]

    env = os.environ.copy()
    if name == "tirex":
        # Avoid upstream xLSTM's CUDA-toolchain import path on GPU hosts.
        env["CUDA_VISIBLE_DEVICES"] = ""

    started = time.perf_counter()
    completed = subprocess.run(cmd, check=True, env=env, text=True, capture_output=True)
    wall_seconds = time.perf_counter() - started
    if completed.stdout.strip():
        print(f"[{name}] {completed.stdout[-1200:]}")
    meta = json.loads(metadata.read_text(encoding="utf-8"))
    meta["wall_seconds"] = wall_seconds
    metadata.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return pd.read_csv(output, parse_dates=["timestamp", "forecast_origin"]), meta

validation_model_frames = []
validation_runtime = {}
for name in SELECTED_MODELS:
    print("validation:", name)
    forecast_frame, meta = run_model(
        name, "validation", WORK_ROOT / "requests" / "validation_history.csv", VALIDATION_HORIZON
    )
    validation_model_frames.append(forecast_frame)
    validation_runtime[name] = meta

validation_models = pd.concat(validation_model_frames, ignore_index=True)
validation_all = pd.concat(
    [validation_models, validation_last, validation_seasonal],
    ignore_index=True,
)

validation_scored, validation_per_series, validation_aggregate = evaluate_forecasts(
    validation_all, validation_truth, validation_seasonal
)

validation_scored.to_csv(OUTPUT_ROOT / "validation" / "forecasts_with_truth.csv", index=False)
validation_per_series.to_csv(OUTPUT_ROOT / "validation" / "per_series_metrics.csv", index=False)
validation_aggregate.to_csv(OUTPUT_ROOT / "validation" / "aggregate_metrics.csv", index=False)
(OUTPUT_ROOT / "validation" / "runtime.json").write_text(
    json.dumps(validation_runtime, indent=2), encoding="utf-8"
)

display(validation_aggregate)
display(validation_per_series)

validation: tirex


CalledProcessError: Command '['work/envs/tirex/bin/python', 'work/runners/tirex_runner.py', '--input', 'work/requests/validation_history.csv', '--output', 'outputs/validation/tirex.csv', '--metadata', 'work/run_metadata/validation_tirex.json', '--cache', 'work/model_cache/tirex', '--horizon', '24', '--partition', 'validation']' returned non-zero exit status 1.

### Read the validation results carefully

The important comparison is not merely which model has the smallest pooled MAE.

Inspect:

- each individual series;
- seasonal-naive performance;
- `MAE skill vs seasonal`;
- q10–q90 coverage;
- interval width; and
- resource cost.

A negative seasonal skill means the foundation model produced **more MAE than simply repeating the previous 24-hour pattern**.

In [ ]:
# @title Plot validation forecasts for one series
PLOT_SERIES = panel["series_ids"][0]

history_plot = validation_history[validation_history["series_id"] == PLOT_SERIES]
truth_plot = validation_truth[validation_truth["series_id"] == PLOT_SERIES]
forecast_plot = validation_all[validation_all["series_id"] == PLOT_SERIES]

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(history_plot["timestamp"], history_plot["target"], label="history")
ax.plot(truth_plot["timestamp"], truth_plot["target"], label="truth")
for model, block in forecast_plot.groupby("model", sort=False):
    ax.plot(block["timestamp"], block["prediction"], label=model)
ax.set_title(f"Validation forecast — series {PLOT_SERIES}")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Target")
ax.legend(ncol=2)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "figures" / "validation_forecasts.png", dpi=150, bbox_inches="tight")
plt.show()

# Show the probabilistic band for each foundation model separately.
for model, block in validation_models[validation_models["series_id"] == PLOT_SERIES].groupby("model", sort=False):
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(truth_plot["timestamp"], truth_plot["target"], label="truth")
    ax.plot(block["timestamp"], block["prediction"], label="q0.5 median")
    ax.fill_between(block["timestamp"], block["q0.1"], block["q0.9"], alpha=0.2, label="q0.1–q0.9")
    ax.set_title(f"{model} — validation predictive quantiles")
    ax.set_xlabel("Timestamp")
    ax.set_ylabel("Target")
    ax.legend()
    plt.tight_layout()
    safe = model.lower().replace(" ", "_").replace(".", "")
    plt.savefig(OUTPUT_ROOT / "figures" / f"validation_{safe}_quantiles.png", dpi=150, bbox_inches="tight")
    plt.show()

## 10. Freeze the experiment before opening the test period

The next cell records the experiment configuration **before** any model sees the final test horizon.

The frozen record includes:

- sample identity;
- selected models;
- immutable revisions;
- environment pins;
- quantiles;
- validation/test horizons;
- seasonal period;
- adapter source digests;
- common context ceilings; and
- validation metrics.

The test section reads and verifies this file instead of redefining the experiment.

In [ ]:
# @title Freeze model/evaluation choices
def source_sha(name):
    return hashlib.sha256(
        (WORK_ROOT / "runners" / f"{name}_runner.py").read_bytes()
    ).hexdigest()

frozen = {
    "notebook_spec": "2.1",
    "profile": "TASK-INFERENCE",
    "mode": "WORKSHOP",
    "sample_sha256": input_sha256,
    "sample_kind": sample_kind,
    "execution_tier": WORKSHOP_TIER,
    "selected_models": SELECTED_MODELS,
    "models": {
        name: {
            "model_id": MODEL_SPECS[name]["model_id"],
            "revision": MODEL_SPECS[name]["revision"],
            "runtime_package": MODEL_SPECS[name]["runtime_package"],
            "runner_sha256": source_sha(name),
        }
        for name in SELECTED_MODELS
    },
    "quantiles": list(QUANTILES),
    "validation_horizon": VALIDATION_HORIZON,
    "test_horizon": TEST_HORIZON,
    "season_length": SEASON_LENGTH,
    "common_min_context": COMMON_MIN_CONTEXT,
    "common_max_context": COMMON_MAX_CONTEXT,
    "common_max_horizon": COMMON_MAX_HORIZON,
    "evaluation": {
        "point_forecast": "q0.5 model median",
        "metrics": [
            "MAE", "RMSE", "MAE skill vs seasonal naive",
            "pinball q0.1/q0.5/q0.9", "q10-q90 coverage", "mean interval width"
        ],
        "baselines": ["last value", f"seasonal naive ({SEASON_LENGTH})"],
    },
    "validation_metrics": validation_aggregate.to_dict("records"),
}

freeze_path = OUTPUT_ROOT / "frozen" / "frozen_experiment.json"
freeze_path.write_text(json.dumps(frozen, indent=2), encoding="utf-8")
print("Frozen experiment:", freeze_path)

# 11. Independent final test

At the later forecast origin, the former validation period is now observed history.

Therefore the test context legitimately contains more observations than the original validation context:

- validation forecast: history ends before the validation period;
- test forecast: history ends before the final test period.

The final test targets remain hidden from configuration/model selection.

In [ ]:
# @title Verify the frozen configuration, then execute the test forecasts
frozen_loaded = json.loads((OUTPUT_ROOT / "frozen" / "frozen_experiment.json").read_text())
assert frozen_loaded["sample_sha256"] == input_sha256
assert frozen_loaded["execution_tier"] == WORKSHOP_TIER
assert frozen_loaded["selected_models"] == SELECTED_MODELS
assert frozen_loaded["quantiles"] == list(QUANTILES)
assert frozen_loaded["test_horizon"] == TEST_HORIZON
assert frozen_loaded["season_length"] == SEASON_LENGTH
for name in SELECTED_MODELS:
    assert frozen_loaded["models"][name]["revision"] == MODEL_SPECS[name]["revision"]
    assert frozen_loaded["models"][name]["runner_sha256"] == source_sha(name)

test_model_frames = []
test_runtime = {}
for name in SELECTED_MODELS:
    print("test:", name)
    forecast_frame, meta = run_model(
        name, "test", WORK_ROOT / "requests" / "test_history.csv", TEST_HORIZON
    )
    test_model_frames.append(forecast_frame)
    test_runtime[name] = meta

test_models = pd.concat(test_model_frames, ignore_index=True)
test_all = pd.concat([test_models, test_last, test_seasonal], ignore_index=True)

test_scored, test_per_series, test_aggregate = evaluate_forecasts(
    test_all, test_truth, test_seasonal
)
test_scored.to_csv(OUTPUT_ROOT / "test" / "forecasts_with_truth.csv", index=False)
test_per_series.to_csv(OUTPUT_ROOT / "test" / "per_series_metrics.csv", index=False)
test_aggregate.to_csv(OUTPUT_ROOT / "test" / "aggregate_metrics.csv", index=False)
(OUTPUT_ROOT / "test" / "runtime.json").write_text(
    json.dumps(test_runtime, indent=2), encoding="utf-8"
)

display(test_aggregate)
display(test_per_series)

In [ ]:
# @title Final test plots
PLOT_SERIES = panel["series_ids"][0]
history_plot = test_history[test_history["series_id"] == PLOT_SERIES]
truth_plot = test_truth[test_truth["series_id"] == PLOT_SERIES]
forecast_plot = test_all[test_all["series_id"] == PLOT_SERIES]

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(history_plot["timestamp"], history_plot["target"], label="history")
ax.plot(truth_plot["timestamp"], truth_plot["target"], label="truth")
for model, block in forecast_plot.groupby("model", sort=False):
    ax.plot(block["timestamp"], block["prediction"], label=model)
ax.set_title(f"Independent test forecast — series {PLOT_SERIES}")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Target")
ax.legend(ncol=2)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "figures" / "test_forecasts.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Computational tradeoffs

Predictive error is only one engineering dimension.

The next table combines measured model load/inference time with known checkpoint size.

Questions to consider:

- Did a foundation model beat seasonal naive on both series?
- If two models are close in error, does one have a much smaller footprint?
- If Toto improves the forecast in `FULL` mode, is the improvement worth a 9.8 GB checkpoint and GPU requirement for this workload?
- Would the answer change for hundreds or thousands of series?

There is no universal engineering winner.

In [ ]:
# @title Compare model footprint and measured test runtime
tradeoff_rows = []
for name in SELECTED_MODELS:
    meta = test_runtime[name]
    tradeoff_rows.append({
        "model": MODEL_SPECS[name]["display_name"],
        "checkpoint_MiB": round(MODEL_SPECS[name]["weight_bytes"] / 2**20, 1),
        "device": meta.get("device"),
        "load_seconds": meta.get("load_seconds"),
        "inference_seconds": meta.get("inference_seconds"),
        "wall_seconds": meta.get("wall_seconds"),
    })
tradeoffs = pd.DataFrame(tradeoff_rows)
display(tradeoffs)
tradeoffs.to_csv(OUTPUT_ROOT / "test" / "computational_tradeoffs.csv", index=False)

## 13. Forecast beyond the observed sample

The entire available panel now becomes context.

This forecast has **no future truth** and is therefore marked **not measurable yet**.

This section separates two ideas:

- **backtesting**: historical future values exist and can be scored;
- **real future inference**: the forecast is available now, but its truth has not occurred yet.

In [ ]:
# @title Generate one 24-step future forecast from all available observations
future_frames = []
future_runtime = {}
for name in SELECTED_MODELS:
    print("future:", name)
    forecast_frame, meta = run_model(
        name, "future", WORK_ROOT / "requests" / "future_history.csv", future_horizon
    )
    future_frames.append(forecast_frame)
    future_runtime[name] = meta

future_forecasts = pd.concat(future_frames, ignore_index=True)
future_forecasts.to_csv(OUTPUT_ROOT / "future" / "forecasts.csv", index=False)
(OUTPUT_ROOT / "future" / "runtime.json").write_text(
    json.dumps(future_runtime, indent=2), encoding="utf-8"
)

display(future_forecasts.head(12))
print("Evidence status: not-measurable — no future truth exists yet.")

# 14. Bring Your Own Data

BYOD uses the same path as the built-in fixture when either:

- `BYOD_CSV_PATH` is populated; or
- `USE_BYOD=True` and one file is interactively selected.

Requirements:

- columns: `series_id`, `timestamp`, `target`;
- unique `(series_id, timestamp)` rows;
- fixed-width regular sampling;
- identical timestamp grid across series;
- finite numerical targets;
- no missing target observations;
- enough history for the requested validation/test tails and at least 32 context points.

The workshop intentionally does **not**:

- impute missing values;
- interpolate gaps;
- resample;
- silently coerce irregular calendars; or
- use future target values as covariates.

### Data movement and privacy

User-supplied data are processed inside the selected notebook runtime and are not submitted to a DIMER worker or DIMER API.

A hosted notebook is still an external compute environment. Do not upload confidential, personal, regulated, security-sensitive, restricted, or proprietary telemetry unless you are authorized to process it there.

# 15. Optional experiments

These exercises are intentionally separate from the frozen canonical test.

### A. Context length

On validation data only, compare shorter historical contexts such as:

- 32 hours;
- 48 hours;
- 72 hours.

Ask:

> Does more historical context always improve the forecast?

Do not use the independent test period to select context length.

### B. Structural break

Generate an additional deterministic series containing:

- trend;
- daily seasonality;
- a sudden late level shift.

Ask:

> Does foundation-model pretraining remove regime-change risk?

### C. Covariates

TiRex-2 and Chronos-2 support covariate-informed forecasting.

An advanced extension may demonstrate genuinely known-future variables such as calendar features.

It must remain outside the common comparison because Toto's current open DIMER capability does not expose an equivalent covariate interface.

A future target—or any feature computed from future target values—is leakage and must be refused.

# 16. Interpretation and limitations

## Synthetic data

The built-in sample is deliberately simple and deterministic. It is suitable for teaching forecasting mechanics and verifying execution, not for ranking models generally.

## Baselines matter

The 24-hour seasonal naive baseline has unusually strong prior information for this fixture because the series were constructed with 24-hour seasonality.

If a foundation model does not improve on it, that does not mean the model is globally weak. It means additional model complexity was not justified by this particular sample and forecast origin.

## Quantiles are model outputs, not guarantees

q0.1 and q0.9 describe the model's predictive distribution.

They are not guaranteed to contain the truth 80% of the time on a new domain. Calibration must be assessed over representative historical forecast origins.

## One validation and one test origin are not a benchmark

Real forecasting evaluation should usually include multiple rolling or blocked origins across operationally representative regimes.

## Distribution shift remains

Real series can contain:

- missing observations;
- sensor or ETL changes;
- structural breaks;
- promotions or interventions;
- incidents and outages;
- changing policies;
- price changes;
- novel products;
- unusual weather;
- demand regime changes.

The built-in fixture represents almost none of these.

## Decision boundary

Forecasts should not autonomously drive high-consequence decisions without representative backtesting, monitoring, fallback procedures, and appropriate human/domain oversight.

## 17. Export provenance and workshop report

In [ ]:
# @title Write final provenance and report bundle
import datetime
import shutil

def file_sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

model_manifest = {
    name: {
        "display_name": MODEL_SPECS[name]["display_name"],
        "model_id": MODEL_SPECS[name]["model_id"],
        "revision": MODEL_SPECS[name]["revision"],
        "license": MODEL_SPECS[name]["license"],
        "runtime_package": MODEL_SPECS[name]["runtime_package"],
        "files": {
            filename: {"bytes": size, "sha256": digest}
            for filename, (size, digest) in MODEL_SPECS[name]["files"].items()
        },
        "runner_sha256": source_sha(name),
    }
    for name in SELECTED_MODELS
}

experiment_manifest = {
    "notebook_spec": "2.1",
    "notebook_profile": "TASK-INFERENCE",
    "notebook_mode": "WORKSHOP",
    "workshop_revision": "0.1.0-candidate",
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "execution_tier": WORKSHOP_TIER,
    "dataset": dataset_manifest,
    "protocol": {
        "validation_context_steps": int(
            validation_history.groupby("series_id").size().min()
        ),
        "validation_horizon": VALIDATION_HORIZON,
        "test_context_steps": int(test_history.groupby("series_id").size().min()),
        "test_horizon": TEST_HORIZON,
        "season_length": SEASON_LENGTH,
        "quantiles": list(QUANTILES),
        "point_forecast": "q0.5 model median",
    },
    "models": model_manifest,
    "test_metrics": test_aggregate.to_dict("records"),
    "standalone_contract": {
        "repository_clone_required": False,
        "dimer_source_runtime_fetch_required": False,
        "external_dimer_worker_required": False,
        "credential_required": False,
        "default_upload_required": False,
    },
    "evidence_scope": (
        "Synthetic tutorial sample; one validation origin and one independent test origin. "
        "Results are sample-sanity evidence, not a model benchmark."
    ),
}

(OUTPUT_ROOT / "provenance" / "model_manifest.json").write_text(
    json.dumps(model_manifest, indent=2), encoding="utf-8"
)
(OUTPUT_ROOT / "provenance" / "experiment_manifest.json").write_text(
    json.dumps(experiment_manifest, indent=2), encoding="utf-8"
)

summary = {
    "tier": WORKSHOP_TIER,
    "models": [MODEL_SPECS[name]["display_name"] for name in SELECTED_MODELS],
    "sample_sha256": input_sha256,
    "test_metrics": test_aggregate.to_dict("records"),
    "future_forecast_status": "not-measurable",
}
(OUTPUT_ROOT / "workshop_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)

bundle = shutil.make_archive(
    str(Path(OUTPUT_DIR).resolve()) + "_DIMER_TimeSeries_FM_Workshop_Report",
    "zip",
    root_dir=Path(OUTPUT_DIR).resolve(),
)

print({
    "output_directory": str(OUTPUT_ROOT.resolve()),
    "report_bundle": bundle,
    "report_bundle_sha256": file_sha(bundle),
})

# 18. Troubleshooting

| What you see | Likely cause | Corrective action |
|---|---|---|
| parent Python older than 3.10 | unsupported notebook kernel | select a current Colab/Kaggle runtime; the model environments use their own pinned Python 3.12 |
| `uv venv --python 3.12` fails | the runtime cannot download a standalone CPython 3.12 | rerun from a connection with internet access, or use a runtime that already provides Python 3.12 |
| virtual-environment install fails | package/network interruption | rerun the environment cell from a clean connection |
| TiRex fails while CUDA is visible | xLSTM tries to resolve a CUDA toolchain | use the workshop runner; it hides CUDA for TiRex CPU execution |
| Chronos snapshot hash fails | incomplete or changed model asset | remove the cached Chronos snapshot and rerun; do not bypass the digest |
| Toto says CUDA unavailable | `FULL` tier selected on CPU | use a CUDA GPU runtime or switch to `STANDARD` |
| Toto download exhausts disk | 9.8 GB checkpoint plus package/cache footprint | use `STANDARD` or a runtime with sufficient storage |
| BYOD timestamps rejected | irregular/gappy calendar | clean the source outside the notebook and document the transformation; the workshop will not silently resample it |
| BYOD target rejected | missing/non-numeric/non-finite values | provide finite numerical observations under the documented contract |
| seasonal baseline fails | history shorter than the seasonal period | use a valid shorter season or more history |
| model loses to seasonal naive | simple structure dominates the sample | treat this as a valid experimental result, not an error |

An explicit refusal is preferable to silently changing the forecasting problem.

# Glossary

| Term | Meaning in this workshop |
|---|---|
| **Context** | Historical observations available when a forecast is made |
| **Forecast horizon** | Number of future timesteps predicted |
| **Forecast origin** | Last observed timestamp before the predicted future |
| **Zero-shot forecasting** | Forecasting with pretrained weights and no task-specific training |
| **Foundation model** | Broadly pretrained model intended to transfer across many series/domains |
| **Last-value baseline** | Repeats the final historical observation |
| **Seasonal naive** | Repeats the corresponding values from the previous seasonal cycle |
| **MAE** | Mean absolute error in target units |
| **RMSE** | Root mean squared error; weights large misses more strongly |
| **MAE skill** | Improvement relative to seasonal-naive MAE |
| **Quantile forecast** | Forecast level below which the model assigns a stated fraction of its predictive distribution |
| **Pinball loss** | Quantile-specific forecast loss |
| **Coverage** | Fraction of observed truth falling inside a quantile band |
| **Interval width** | Average distance between lower and upper forecast quantiles |
| **Validation period** | Future held out for comparison/configuration before test |
| **Test period** | Independent future held out until the experiment is frozen |
| **Leakage** | Future information reaching the model or model-selection process |
| **Distribution shift** | Change between historical/training conditions and the future being forecast |

In [ ]:
# @title Run-all completion summary
completion = {
    "notebook_spec": "2.1",
    "profile": "TASK-INFERENCE",
    "mode": "WORKSHOP",
    "tier": WORKSHOP_TIER,
    "models": [MODEL_SPECS[name]["display_name"] for name in SELECTED_MODELS],
    "sample_kind": sample_kind,
    "sample_sha256": input_sha256,
    "validation_context": int(validation_history.groupby("series_id").size().min()),
    "validation_horizon": VALIDATION_HORIZON,
    "test_context": int(test_history.groupby("series_id").size().min()),
    "test_horizon": TEST_HORIZON,
    "future_horizon": future_horizon,
    "output_directory": str(OUTPUT_ROOT.resolve()),
}
display(pd.Series(completion, name="value").to_frame())

print(
    "Canonical workshop path complete. "
    "Treat all built-in-sample metrics as tutorial/sanity evidence, not benchmark evidence."
)